# 3. Longer sequences, and what changes

PARNET was trained on 600 nt windows, but nothing in the architecture fixes the length.
Every layer is `stride=1, padding='same'`, so you can feed a whole transcript and get a
prediction the same length back.

Whether you *should* is a different question, and this notebook measures the two things
that actually change: the receptive field relative to the input, and the scale of the
numbers coming out.

In [1]:
import torch

torch.set_num_threads(2)

import numpy as np
import parnet_tutorials as pt
from parnet.models import RBPNet

MODEL = 'parnet-21m-penalty5'  # change this one line to run the notebook on another model
model = RBPNet.from_pretrained(pt.model_path(MODEL))
arch = model.parnet_config['architecture']
print('body blocks:', arch['n_body_blocks'])
print('dilations  :', arch['body']['dilations'])

/home/user/projects/parnet-project/parnet--analyses--interpretation/.pixi/envs/parnet-dev-cu12/lib/python3.10/site-packages/torch/nn/modules/conv.py:370: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1025.)
  return F.conv1d(


body blocks: 12
dilations  : [1, 1, 3, 5, 9, 16, 28, 50, 87, 153, 269, 471]


## Receptive field

Each body block is a kernel-3 convolution at its own dilation, so it adds
`2 * dilation` positions of context. The stem adds `kernel_size - 1`.

This is arithmetic, not a measurement, but it is the number that decides whether a window
length makes sense.

In [2]:
stem_span = arch['stem']['kernel_size'] - 1
body_span = sum(2 * d for d in arch['body']['dilations'])
receptive_field = 1 + stem_span + body_span

print(f'stem contributes      {stem_span:5d} nt')
print(f'body contributes      {body_span:5d} nt')
print(f'receptive field       {receptive_field:5d} nt')
print()
for length in (300, 600, 2000, 10000):
    covered = min(receptive_field, length) / length
    note = 'every position sees the whole window' if receptive_field >= length else ''
    print(f'  window {length:6d} nt: receptive field covers {covered:5.1%}  {note}')

stem contributes         11 nt
body contributes       2186 nt
receptive field        2198 nt

  window    300 nt: receptive field covers 100.0%  every position sees the whole window
  window    600 nt: receptive field covers 100.0%  every position sees the whole window
  window   2000 nt: receptive field covers 100.0%  every position sees the whole window
  window  10000 nt: receptive field covers 22.0%  


At 600 nt the receptive field exceeds the window, so every output position can see every
input position. At 10,000 nt it cannot, and a position near one end knows nothing about
the other. That is not automatically wrong, but it is a different regime from the one the
model was trained in.

## What actually changes in the output

Run the same 600 nt of sequence alone, then embedded in longer and longer flanks, and
compare the prediction over the shared region.

Because each track is normalised over positions, a longer window spreads the same
probability mass over more positions, so densities fall simply because there are more
places for them to go. The comparison that survives that is the **shape** over the shared
region, so compare correlations, not values.

In [3]:
rng = np.random.default_rng(0)
core = ''.join(rng.choice(list('ACGT'), size=600))
rbp = pt.demo_track(MODEL)
task = pt.track_index(MODEL, rbp)

def predict(sequence):
    from parnet.tokenization import RnaTokenizer
    with torch.no_grad():
        return model(**RnaTokenizer()(sequence))

reference = predict(core)['total'][0, task].exp().numpy()

rows = []
for flank in (0, 200, 700, 1700):
    left = ''.join(rng.choice(list('ACGT'), size=flank))
    right = ''.join(rng.choice(list('ACGT'), size=flank))
    out = predict(left + core + right)['total'][0, task].exp().numpy()
    shared = out[flank:flank + 600]
    rows.append({
        'total length': 600 + 2 * flank,
        'flank': flank,
        'peak density in core': shared.max(),
        'mass in core': shared.sum(),
        'shape PCC vs 600 nt': np.corrcoef(shared, reference)[0, 1],
    })

import pandas as pd
pd.set_option('display.width', 200)
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f'{v:.4f}'))

 total length  flank  peak density in core  mass in core  shape PCC vs 600 nt
          600      0                0.2083        5.4386               1.0000
         1000    200                0.1040        2.7496               0.9815
         2000    700                0.0798        2.0742               0.9861
         4000   1700                0.0394        1.1058               0.9834


Two things to read off that table. The mass inside the core falls as the window grows,
which is normalisation doing its job and says nothing about the biology. The shape
correlation is the honest comparison.

## The practical advice

Predicting a long transcript in one call is legitimate, but it answers a different
question from the one the model was trained on, and the densities are not comparable
across lengths.

If you want 600 nt behaviour over a long region, tile it. Tiles overlap, but only the
central half of each is kept, so every position is predicted exactly once, by the tile in
which it sits most centrally. That is the position with the most context on both sides,
which is what avoids edge effects.

In [4]:
def tile(sequence, window=600, stride=300):
    """Yield (start, subsequence) pairs covering `sequence` with overlap."""
    for start in range(0, max(1, len(sequence) - window + 1), stride):
        yield start, sequence[start:start + window]


long_sequence = ''.join(rng.choice(list('ACGT'), size=3000))
tiles = list(tile(long_sequence))
print(f'{len(long_sequence)} nt -> {len(tiles)} tiles of 600 nt at stride 300')

from parnet.tokenization import RnaTokenizer

with torch.no_grad():
    predictions = model(**RnaTokenizer()([s for _, s in tiles]))['total'][:, task].exp().numpy()

# Keep the central half of each tile, which is the part with context on both sides.
stitched = np.zeros(len(long_sequence))
counts = np.zeros(len(long_sequence))
keep = slice(150, 450)
for (start, _), prediction in zip(tiles, predictions):
    stitched[start + keep.start:start + keep.stop] += prediction[keep]
    counts[start + keep.start:start + keep.stop] += 1

covered = counts > 0
print(f'covered {covered.sum()} of {len(long_sequence)} positions')
# Kept regions are 300 nt wide at stride 300, so they abut exactly: every covered
# position comes from one tile, the one it is most central in. No averaging seam.
print(f'times each covered position was predicted: min {counts[covered].min():.0f}, '
      f'max {counts[covered].max():.0f}')
stitched[covered] /= counts[covered]
print(f'stitched profile: peak {stitched.max():.3e} at position {stitched.argmax()}')

3000 nt -> 9 tiles of 600 nt at stride 300


covered 2700 of 3000 positions
times each covered position was predicted: min 1, max 1
stitched profile: peak 1.512e+00 at position 354


## What to take away

1. Any length runs. Nothing in the architecture fixes 600 nt.
2. The receptive field here is the number that decides whether a length is sensible.
3. Densities are normalised per window, so they are not comparable across lengths.
   Compare shapes.
4. For long regions, tiling with overlap and keeping tile centres stays in the regime the
   model was trained on.